# 🏏 IPL First Innings Score Predictor

**A Machine Learning project to predict the final score of the batting team in an IPL match.**

---
| Stage | Details |
|---|---|
| **Training Data** | IPL Seasons 1–9 (2008–2016) |
| **Test Data** | IPL Season 10 (2017) |
| **Prediction** | IPL Seasons 11–12 (2018–2019) |
| **Best Model** | Linear Regression + AdaBoost |

## 📦 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Sklearn
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, AdaBoostRegressor
from sklearn.metrics import mean_absolute_error as mae, mean_squared_error as mse

print('✅ All libraries imported successfully!')

## 📂 2. Load Dataset

In [ ]:
df = pd.read_csv('/kaggle/input/datasets/taiwangirl/ipl-dataset/ipl.csv')
print(f'✅ Dataset loaded! Shape: {df.shape}')
df.head()

## 🔍 3. Exploratory Data Analysis

In [ ]:
print('📊 Dataset Info')
print('='*50)
print(f'Shape       : {df.shape}')
print(f'Columns     : {list(df.columns)}')
print(f'\nData Types:')
print(df.dtypes)
print(f'\nMissing Values:')
print(df.isnull().sum())

In [ ]:
# Distribution of total scores
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('📊 IPL Score Distribution', fontsize=16, fontweight='bold')

axes[0].hist(df['total'], bins=40, color='#1f77b4', edgecolor='white', alpha=0.85)
axes[0].set_title('Distribution of Total Scores')
axes[0].set_xlabel('Total Score')
axes[0].set_ylabel('Frequency')
axes[0].axvline(df['total'].mean(), color='red', linestyle='--', label=f'Mean: {df["total"].mean():.1f}')
axes[0].legend()

top_teams = df.groupby('bat_team')['total'].mean().sort_values(ascending=False).head(10)
top_teams.plot(kind='barh', ax=axes[1], color='#2ecc71', edgecolor='white')
axes[1].set_title('Avg Score by Batting Team')
axes[1].set_xlabel('Average Total Score')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

## 🧹 4. Data Cleaning

In [ ]:
# --- Step 1: Remove unwanted columns ---
columns_to_remove = ['mid', 'venue', 'batsman', 'bowler', 'striker', 'non-striker']
print(f'Before removing unwanted columns : {df.shape}')
df.drop(labels=columns_to_remove, axis=1, inplace=True)
print(f'After removing unwanted columns  : {df.shape}')

# --- Step 2: Keep only consistent teams ---
consistent_teams = [
    'Kolkata Knight Riders', 'Chennai Super Kings', 'Rajasthan Royals',
    'Mumbai Indians', 'Kings XI Punjab', 'Royal Challengers Bangalore',
    'Delhi Daredevils', 'Sunrisers Hyderabad'
]
print(f'\nBefore removing inconsistent teams : {df.shape}')
df = df[(df['bat_team'].isin(consistent_teams)) & (df['bowl_team'].isin(consistent_teams))]
print(f'After removing inconsistent teams  : {df.shape}')

# --- Step 3: Remove first 5 overs (powerplay) ---
print(f'\nBefore removing first 5 overs data : {df.shape}')
df = df[df['overs'] >= 5.0]
print(f'After removing first 5 overs data  : {df.shape}')

# --- Step 4: Convert date column ---
df['date'] = pd.to_datetime(df['date'], format='%Y-%m-%d')
print(f'\n✅ Date column type: {type(df.iloc[0, 0])}')

## 🔥 5. Correlation Heatmap

In [ ]:
numeric_df = df.select_dtypes(include=['int64', 'float64'])
corr_matrix = numeric_df.corr()

plt.figure(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix, annot=True, fmt='.2f',
    cmap='RdYlGn', mask=mask,
    linewidths=0.5, square=True,
    cbar_kws={'shrink': 0.8}
)
plt.title('🔥 Feature Correlation Heatmap', fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

## ⚙️ 6. Data Preprocessing

In [ ]:
# One-Hot Encoding
encoded_df = pd.get_dummies(data=df, columns=['bat_team', 'bowl_team'])

# Rearrange columns
feature_cols = [
    'date',
    'bat_team_Chennai Super Kings', 'bat_team_Delhi Daredevils', 'bat_team_Kings XI Punjab',
    'bat_team_Kolkata Knight Riders', 'bat_team_Mumbai Indians', 'bat_team_Rajasthan Royals',
    'bat_team_Royal Challengers Bangalore', 'bat_team_Sunrisers Hyderabad',
    'bowl_team_Chennai Super Kings', 'bowl_team_Delhi Daredevils', 'bowl_team_Kings XI Punjab',
    'bowl_team_Kolkata Knight Riders', 'bowl_team_Mumbai Indians', 'bowl_team_Rajasthan Royals',
    'bowl_team_Royal Challengers Bangalore', 'bowl_team_Sunrisers Hyderabad',
    'overs', 'runs', 'wickets', 'runs_last_5', 'wickets_last_5', 'total'
]

# Only keep cols that exist in the dataframe
existing_cols = [c for c in feature_cols if c in encoded_df.columns]
encoded_df = encoded_df[existing_cols]

print(f'✅ Encoded dataframe shape: {encoded_df.shape}')
encoded_df.head(3)

In [ ]:
# Train-Test Split based on year
X_train = encoded_df.drop(labels='total', axis=1)[encoded_df['date'].dt.year <= 2016]
X_test  = encoded_df.drop(labels='total', axis=1)[encoded_df['date'].dt.year >= 2017]

y_train = encoded_df[encoded_df['date'].dt.year <= 2016]['total'].values
y_test  = encoded_df[encoded_df['date'].dt.year >= 2017]['total'].values

X_train.drop(labels='date', axis=1, inplace=True)
X_test.drop(labels='date', axis=1, inplace=True)

# Save column order for predictions
X_train_cols = X_train.columns.tolist()

print(f'Training set : {X_train.shape}')
print(f'Test set     : {X_test.shape}')
print(f'y_train      : {y_train.shape}')
print(f'y_test       : {y_test.shape}')

## 🤖 7. Model Building & Evaluation

In [ ]:
results = {}

# ── Linear Regression ──────────────────────────────────────────
lr = LinearRegression()
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)
results['Linear Regression'] = {
    'MAE':  mae(y_test, y_pred_lr),
    'MSE':  mse(y_test, y_pred_lr),
    'RMSE': np.sqrt(mse(y_test, y_pred_lr))
}

# ── Decision Tree ───────────────────────────────────────────────
dt = DecisionTreeRegressor(random_state=42)
dt.fit(X_train, y_train)
y_pred_dt = dt.predict(X_test)
results['Decision Tree'] = {
    'MAE':  mae(y_test, y_pred_dt),
    'MSE':  mse(y_test, y_pred_dt),
    'RMSE': np.sqrt(mse(y_test, y_pred_dt))
}

# ── Random Forest ───────────────────────────────────────────────
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
results['Random Forest'] = {
    'MAE':  mae(y_test, y_pred_rf),
    'MSE':  mse(y_test, y_pred_rf),
    'RMSE': np.sqrt(mse(y_test, y_pred_rf))
}

# ── AdaBoost (on Linear Regression) ────────────────────────────
adb = AdaBoostRegressor(estimator=LinearRegression(), n_estimators=100, random_state=42)
adb.fit(X_train, y_train)
y_pred_adb = adb.predict(X_test)
results['AdaBoost (LR)'] = {
    'MAE':  mae(y_test, y_pred_adb),
    'MSE':  mse(y_test, y_pred_adb),
    'RMSE': np.sqrt(mse(y_test, y_pred_adb))
}

# Print results table
results_df = pd.DataFrame(results).T.round(2)
print('\n📊 Model Comparison:')
print('='*55)
print(results_df.to_string())
print('='*55)

best_model_name = results_df['RMSE'].idxmin()
print(f'\n🏆 Best model by RMSE: {best_model_name}')

In [ ]:
# Model comparison bar chart
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('📊 Model Performance Comparison', fontsize=15, fontweight='bold')

colors = ['#3498db', '#e74c3c', '#2ecc71', '#f39c12']
metrics = ['MAE', 'MSE', 'RMSE']

for idx, metric in enumerate(metrics):
    vals = [results[m][metric] for m in results]
    bars = axes[idx].bar(results.keys(), vals, color=colors, edgecolor='white', linewidth=1.2)
    axes[idx].set_title(f'{metric}', fontsize=12, fontweight='bold')
    axes[idx].set_ylabel(metric)
    axes[idx].tick_params(axis='x', rotation=20)
    for bar, val in zip(bars, vals):
        axes[idx].text(bar.get_x() + bar.get_width()/2., bar.get_height() + max(vals)*0.01,
                       f'{val:.2f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Actual vs Predicted scatter plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('🎯 Actual vs Predicted Scores', fontsize=14, fontweight='bold')

for ax, (preds, label, color) in zip(axes, [
    (y_pred_lr, 'Linear Regression', '#3498db'),
    (y_pred_adb, 'AdaBoost (LR)', '#f39c12')
]):
    ax.scatter(y_test, preds, alpha=0.3, color=color, s=15)
    mn, mx = y_test.min(), y_test.max()
    ax.plot([mn, mx], [mn, mx], 'r--', linewidth=2, label='Perfect Fit')
    ax.set_xlabel('Actual Score')
    ax.set_ylabel('Predicted Score')
    ax.set_title(label)
    ax.legend()

plt.tight_layout()
plt.show()

## 🔮 8. Prediction Function

In [ ]:
# Use Linear Regression as the final model (best RMSE among simple models)
final_model = lr

def predict_score(batting_team, bowling_team, overs, runs, wickets,
                  runs_in_prev_5, wickets_in_prev_5):
    """Predict the final first innings score given match state."""
    
    # Build a row that matches training encoding
    row = {col: 0 for col in X_train_cols}
    row['overs']          = overs
    row['runs']           = runs
    row['wickets']        = wickets
    row['runs_last_5']    = runs_in_prev_5
    row['wickets_last_5'] = wickets_in_prev_5
    
    bat_col  = f'bat_team_{batting_team}'
    bowl_col = f'bowl_team_{bowling_team}'
    if bat_col in row:  row[bat_col]  = 1
    if bowl_col in row: row[bowl_col] = 1
    
    input_df = pd.DataFrame([row])
    prediction = int(final_model.predict(input_df)[0])
    return prediction

print('✅ predict_score() function ready!')

## 📌 9. Sample Predictions

In [ ]:
test_cases = [
    dict(batting_team='Kolkata Knight Riders', bowling_team='Delhi Daredevils',
         overs=15.2, runs=122, wickets=3, runs_in_prev_5=45, wickets_in_prev_5=1,
         actual=200, label='KKR vs DD — 16 Apr 2018'),
    dict(batting_team='Sunrisers Hyderabad', bowling_team='Royal Challengers Bangalore',
         overs=13.4, runs=103, wickets=6, runs_in_prev_5=32, wickets_in_prev_5=2,
         actual=146, label='SRH vs RCB — 7 May 2018'),
    dict(batting_team='Mumbai Indians', bowling_team='Kings XI Punjab',
         overs=14.1, runs=136, wickets=4, runs_in_prev_5=50, wickets_in_prev_5=0,
         actual=186, label='MI vs KXIP — 17 May 2018'),
    dict(batting_team='Mumbai Indians', bowling_team='Kings XI Punjab',
         overs=12.3, runs=113, wickets=2, runs_in_prev_5=55, wickets_in_prev_5=0,
         actual=176, label='MI vs KXIP — 30 Mar 2019'),
    dict(batting_team='Rajasthan Royals', bowling_team='Chennai Super Kings',
         overs=13.3, runs=92, wickets=5, runs_in_prev_5=27, wickets_in_prev_5=2,
         actual=151, label='RR vs CSK — 11 Apr 2019'),
    dict(batting_team='Delhi Daredevils', bowling_team='Sunrisers Hyderabad',
         overs=11.5, runs=98, wickets=3, runs_in_prev_5=41, wickets_in_prev_5=1,
         actual=155, label='DD vs SRH — 14 Apr 2019'),
    dict(batting_team='Delhi Daredevils', bowling_team='Chennai Super Kings',
         overs=10.2, runs=68, wickets=3, runs_in_prev_5=29, wickets_in_prev_5=1,
         actual=147, label='DD vs CSK — 10 May 2019 (Eliminator)'),
]

print('\n🏏 IPL Score Prediction Results')
print('='*75)
print(f'{"Match":<42} {"Actual":>8} {"Predicted":>12} {"Range":>18}')
print('-'*75)

for tc in test_cases:
    pred = predict_score(
        tc['batting_team'], tc['bowling_team'], tc['overs'],
        tc['runs'], tc['wickets'], tc['runs_in_prev_5'], tc['wickets_in_prev_5']
    )
    r = f'{pred-10} – {pred+5}'
    hit = '✅' if tc['actual'] >= pred-10 and tc['actual'] <= pred+5 else '❌'
    print(f'{hit} {tc["label"]:<40} {tc["actual"]:>8} {pred:>12} {r:>18}')

print('='*75)

## 🚀 10. Gradio Deployment

In [ ]:
# Install Gradio if not already installed
import subprocess
subprocess.run(['pip', 'install', 'gradio', '-q'], capture_output=True)
print('✅ Gradio ready!')

In [ ]:
import gradio as gr

TEAMS = [
    'Chennai Super Kings',
    'Delhi Daredevils',
    'Kings XI Punjab',
    'Kolkata Knight Riders',
    'Mumbai Indians',
    'Rajasthan Royals',
    'Royal Challengers Bangalore',
    'Sunrisers Hyderabad'
]

def gradio_predict(batting_team, bowling_team, overs, runs, wickets, runs_last_5, wickets_last_5):
    if batting_team == bowling_team:
        return '⚠️ Batting and Bowling teams cannot be the same!', ''
    if overs < 5.0 or overs > 19.5:
        return '⚠️ Overs must be between 5.0 and 19.5', ''

    predicted = predict_score(
        batting_team, bowling_team, overs,
        runs, wickets, runs_last_5, wickets_last_5
    )

    lo, hi = predicted - 10, predicted + 5
    score_range = f"{lo} – {hi}"

    # Run rate
    curr_rr = runs / overs if overs > 0 else 0
    req_rr  = (predicted - runs) / (20 - overs) if (20 - overs) > 0 else 0

    summary = (
        f"🏏 **Match Summary**\n\n"
        f"- **{batting_team}** batting vs **{bowling_team}**\n"
        f"- Current score: **{runs}/{wickets}** in **{overs:.1f}** overs\n"
        f"- Current Run Rate: **{curr_rr:.2f}**\n"
        f"- Required Run Rate (to reach predicted): **{req_rr:.2f}**\n"
        f"- Last 5 overs: **{runs_last_5} runs, {wickets_last_5} wickets**"
    )

    return score_range, summary

# ── Build Gradio UI ──────────────────────────────────────────────
with gr.Blocks(
    title="🏏 IPL Score Predictor",
    theme=gr.themes.Soft(primary_hue="orange", secondary_hue="blue")
) as demo:

    gr.Markdown("""
    # 🏏 IPL First Innings Score Predictor
    **Predict the final first innings score using a Linear Regression model trained on IPL data (2008–2016).**
    """)

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### 🆚 Teams")
            batting_team  = gr.Dropdown(choices=TEAMS, label="🏏 Batting Team",  value='Mumbai Indians')
            bowling_team  = gr.Dropdown(choices=TEAMS, label="🎯 Bowling Team", value='Chennai Super Kings')

        with gr.Column(scale=1):
            gr.Markdown("### 📊 Current Match State")
            overs    = gr.Slider(minimum=5.0, maximum=19.5, step=0.1, value=10.0, label="🕐 Current Over")
            runs     = gr.Number(value=80,  label="🏃 Runs Scored So Far",  precision=0)
            wickets  = gr.Slider(minimum=0, maximum=9,   step=1,   value=2,  label="❌ Wickets Fallen")

        with gr.Column(scale=1):
            gr.Markdown("### ⚡ Last 5 Overs")
            runs_last_5    = gr.Number(value=40, label="🔥 Runs in Last 5 Overs",    precision=0)
            wickets_last_5 = gr.Slider(minimum=0, maximum=5, step=1, value=1,
                                       label="❌ Wickets in Last 5 Overs")

    predict_btn = gr.Button("🔮  Predict Final Score", variant="primary", size="lg")

    with gr.Row():
        with gr.Column():
            gr.Markdown("### 🎯 Predicted Score Range")
            output_range = gr.Textbox(label="Score Range (predicted ± buffer)", interactive=False,
                                      placeholder="e.g.  155 – 175")
        with gr.Column():
            gr.Markdown("### 📋 Match Insights")
            output_summary = gr.Markdown()

    gr.Markdown("""
    ---
    **ℹ️ Notes:**
    - Model trained on IPL 2008–2016 data; only 8 historic franchises are supported.
    - Predictions are most reliable between overs 10–16.
    - Score range = predicted − 10 to predicted + 5.
    """)

    predict_btn.click(
        fn=gradio_predict,
        inputs=[batting_team, bowling_team, overs, runs, wickets, runs_last_5, wickets_last_5],
        outputs=[output_range, output_summary]
    )

# ── Launch ───────────────────────────────────────────────────────
demo.launch(share=True)